In [1]:
# import and get the pdf 
import warnings 
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader 
# load document  
pdf = PyPDFLoader('GK_Questions.pdf')

try:
    document = pdf.load()
except Exception as e :
    print(str(e)) 

In [2]:
# split Data 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
text_split = RecursiveCharacterTextSplitter(chunk_size=1600 , chunk_overlap = 180)
split_document = text_split.split_documents(document)

chunks = [i.page_content for i in split_document]
metadata = [i.metadata for i in split_document]

#  create unique IDs

import hashlib
ids = [hashlib.md5(chunk.encode('utf-8')).hexdigest() for chunk in chunks]

In [3]:
# Hybrid Create 
from rank_bm25 import BM25Okapi 
token_corpus = BM25Okapi([i.split() for i in chunks])
token_corpus

In [4]:
# Create VectorDB 
from chromadb.utils.embedding_functions import DefaultEmbeddingFunction 
import chromadb 

# Client 
client = chromadb.PersistentClient(path="./VectorDB")

# Collection 
collection = client.get_or_create_collection(name="Collection",embedding_function=DefaultEmbeddingFunction())

try:
    if collection.count()!= len(chunks):
        collection.add(
        ids=ids ,
        documents=chunks , 
        metadatas=metadata
    )
except Exception as e :
    print(str(e))


In [5]:
# LLM Connection  
from langchain_groq import ChatGroq 
from dotenv import load_dotenv 
import os 
load_dotenv()
try:
    key = os.getenv("GROQ_API_KEY")
    print(bool(key))

    # Groq model
    LLM = ChatGroq(model="openai/gpt-oss-120b",api_key=key)
except Exception as e :
    print(str(e))


True


In [6]:
from langchain_core.tools import tool

@tool
def Hybrid_connection(query: str):
    """
    Retrieve relevant information from the local knowledge base
    using hybrid search that combines BM25 keyword retrieval
    with semantic vector similarity search.

    Use this tool when the user's question is related to information
    contained in the stored documents or knowledge base.

    This tool is especially useful when the query contains:
    - specific keywords, names, terms, or phrases
    - exact facts or technical terminology
    - concepts that may require semantic similarity
    - questions where both keyword matching and semantic understanding
      are useful

    Do not use this tool for simple mathematical calculations,
    general conversation, or current/latest information from the web.
    """

    result = collection.query(
        query_texts=[query],
        n_results=3
    )

    document = result['documents'][0]
    distances = result['distances'][0]
    print(distances ,)
    dense_chunks = []
    threshold = 1.9

    for i , doc in zip(distances , document):
        if i < threshold:
            dense_chunks.append(doc)

    score = token_corpus.get_scores(query.split())

    def get_near_keyword(score , k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index , key = lambda x : x [1] , reverse=True)
        return [i for i , _ in index_sorted[:10]]

    keyword_search = get_near_keyword(score , k=10)

    get_chunks = [chunks[i] for i in keyword_search]

    rrf_token_corpus = {}

    for rank , docs in enumerate(dense_chunks):
        rrf_token_corpus[docs] = rrf_token_corpus.get(docs , 0)+1/(rank+60)
    for rank , docs in enumerate(get_chunks):
        rrf_token_corpus[docs] = rrf_token_corpus.get(docs , 0)+1/(rank+60)
    marge = sorted(rrf_token_corpus.items() , key=lambda x:x[1] , reverse=True)
    top_docs = [doc for doc , i in marge[:5]]

    if not top_docs:
        return "NOT RELATED CONTENTS"
    return "\n\n".join(top_docs)


In [7]:
# calculator 
import numexpr 
@tool 
def calculator(execution:str):
    """Perform mathematical calculations from a given expression.

    Use this tool when the user asks for arithmetic or numerical
    calculations such as addition, subtraction, multiplication,
    division, percentages, powers, or other basic mathematical
    expressions.

    Do not use this tool for general questions, document retrieval,
    or current/latest information from the web."""
    try :
        response = numexpr.evaluate(execution)
        return str(response)
    except Exception as e :
        return str(e)

# Web Search Fall Back 
from langchain_community.tools import DuckDuckGoSearchRun
@tool
def web_search(query: str) -> str:
    """
    Search the internet for current, recent, or publicly available information.

    Use this tool when the user asks for:
    - current or latest information
    - recent news or events
    - information that may have changed over time
    - current facts, prices, trends, or updates
    - information that is not available in the local knowledge base

    Do not use this tool for:
    - mathematical calculations; use the calculator tool
    - questions that can be answered from the local document knowledge base;
      use the hybrid_retrieve tool instead
    - general conversation or greetings

    Input:
        query: A clear search query describing the information to find.
    """
    try:
        search = DuckDuckGoSearchRun()
        return search.run()
    except Exception as e :
        return str(e)

## Graph 

In [8]:
from langgraph.graph import StateGraph, START , END 
from typing import TypedDict , Annotated 
from langgraph.graph.message import add_messages 

# State build 
class state(TypedDict):
    messages:Annotated[list,add_messages]

build = StateGraph(state)

#tools

tools = [Hybrid_connection , web_search , calculator]

#tool bind
LLM_bind = LLM.bind_tools(tools)

def llm_tool_bind(state:state):

    response = LLM_bind.invoke(state["messages"])

    return{
        "messages": response
    }

# Node build 
from langgraph.prebuilt import ToolNode , tools_condition 
build.add_node('llm_tools' , llm_tool_bind)
build.add_node('tools',ToolNode(tools))

# Edge 
build.add_edge(START , 'llm_tools')
build.add_conditional_edges(

    'llm_tools',

    tools_condition ,

    {"tools": "tools", "__end__": "__end__"},

)
build.add_edge('tools','llm_tools')

graph = build.compile()

# testing 
from langchain_core.messages import HumanMessage 
response = graph.invoke({
    'messages':
    HumanMessage('what city called city of joy ?')
})

result = response['messages'][-1].content
print(result)

The nickname **“City of Joy”** most famously refers to **Kolkata (formerly Calcutta), India**.  

- The moniker became popular worldwide after Dominique Lapierre’s 1985 book *The City of Joy*, which portrayed life in the city’s slums and highlighted the resilience and spirit of its residents.  
- Kolkata’s rich cultural heritage, vibrant festivals, and warm, welcoming people reinforce the idea of a “city of joy” for both locals and visitors.  

While other places sometimes use similar slogans, Kolkata is the city most universally associated with the title “City of Joy.”


## Evelution 

In [ ]:
from ragas.evaluation import evaluate 
from langchain_google_vertexai import ChatVertexAI
from langchain_google_vertexai import VertexAI
from datasets import Dataset 
from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision, ContextRecall

def Hybrid_connection(query: str):

    result = collection.query(
        query_texts=[query],
        n_results=3
    )

    document = result['documents'][0]
    distances = result['distances'][0]
    print(distances ,)
    dense_chunks = []
    threshold = 1.9

    for i , doc in zip(distances , document):
        if i < threshold:
            dense_chunks.append(doc)

    score = token_corpus.get_scores(query.split())

    def get_near_keyword(score , k=10):
        index = list(enumerate(score))
        index_sorted = sorted(index , key = lambda x : x [1] , reverse=True)
        return [i for i , _ in index_sorted[:10]]

    keyword_search = get_near_keyword(score , k=10)

    get_chunks = [chunks[i] for i in keyword_search]

    rrf_token_corpus = {}

    for rank , docs in enumerate(dense_chunks):
        rrf_token_corpus[docs] = rrf_token_corpus.get(docs , 0)+1/(rank+60)
    for rank , docs in enumerate(get_chunks):
        rrf_token_corpus[docs] = rrf_token_corpus.get(docs , 0)+1/(rank+60)
    marge = sorted(rrf_token_corpus.items() , key=lambda x:x[1] , reverse=True)
    top_docs = [doc for doc , i in marge[:5]]

    return top_docs 

def generation(question: str, context_list: list):

    if not context_list:
        return "NOT RELATED CONTENTS"

    context_str = "\n\n".join(context_list)

    prompt = f"""
You are a helpful AI assistant for a Retrieval-Augmented Generation (RAG) system.

Your task is to answer the user's question using ONLY the information
provided in the retrieved context.

Rules:
1. Use only the given context to answer the question.
2. Do not use outside knowledge or make up information.
3. If the context does not contain enough information to answer the question,
   say: "I don't have enough information in the provided context."
4. Give a clear, accurate, and concise answer.
5. Do not mention the retrieval process unless necessary.
6. If multiple pieces of context are relevant, combine them into one coherent answer.

Retrieved Context:
{context_str}

User Question:
{question}

Final Answer:
"""

    answer = LLM.invoke(prompt)
    return answer.content

Question = "What is the name of India's lower house of Parliament?"
Ground_truth = "Lok Sabha"

retrive = Hybrid_connection(query=Question)
LLM_answer = generation(question=Question , context_list=retrive)

data = {
        "user_input":[Question] , 
        "retrieved_contexts":[retrive] , 
        "response":[LLM_answer] , 
        "reference":[Ground_truth]
    }
dataset = Dataset.from_dict(data)

from langchain_community.embeddings import HuggingFaceEmbeddings 
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

result = evaluate(
    dataset=dataset , 
    embeddings=embeddings , 
    metrics=[Faithfulness(),AnswerRelevancy(),ContextPrecision(),ContextRecall()] , 
    llm=LLM , 
    raise_exceptions=False
)
print(result)


[1.2584388256072998, 1.5907402038574219, 1.7816210985183716]


C:\Users\TOTAN SANTRA\AppData\Local\Temp\ipykernel_19812\2825077348.py:95: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
Evaluating: 100%|██████████| 4/4 [00:53<00:00, 13.46s/it]

{'faithfulness': 1.0000, 'answer_relevancy': 0.9648, 'context_precision': 1.0000, 'context_recall': 1.0000}


: 